In [6]:
from IPython.display import display

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project structure
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"

# Ground-truth file
GT_PATH = RAW_DATA_DIR / "train_ground_truth.tsv"

print("Project root:", PROJECT_ROOT)
print("Ground truth:", GT_PATH)
print("Ground truth exists:", GT_PATH.exists())

Project root: d:\amzn_ml_challenge
Ground truth: d:\amzn_ml_challenge\data\raw\train_ground_truth.tsv
Ground truth exists: True


In [2]:
from IPython import display
gt = pd.read_csv(
    GT_PATH,
    sep="\t"
)

print("Shape:", gt.shape)
print("\nColumns:")
print(gt.columns.tolist())

print(gt.head())

Shape: (2206821, 2)

Columns:
['source1_entity_id', 'matched_entity_ids']
  source1_entity_id                                 matched_entity_ids
0         S1-965667  S2-681193310,S2-743505751,S3-775321672,S3-1129...
1       S1-55344266  S2-249013014,S2-197070651,S3-478195123,S3-3843...
2      S1-343815751             S2-790675320,S2-479876582,S3-878454467
3      S1-656753428              S2-153058913,S2-24659151,S3-679606215
4      S1-102811957  S2-478959098,S2-553508714,S2-625774905,S3-7280...


In [3]:
gt.info()

<class 'pandas.DataFrame'>
RangeIndex: 2206821 entries, 0 to 2206820
Data columns (total 2 columns):
 #   Column              Dtype
---  ------              -----
 0   source1_entity_id   str  
 1   matched_entity_ids  str  
dtypes: str(2)
memory usage: 33.7 MB


In [4]:
print(
    gt.sample(5, random_state=42)
)

        source1_entity_id                                 matched_entity_ids
2195840      S1-384734976  S2-685146894,S2-908470079,S3-109845102,S3-1764...
1487051      S1-574038342             S2-133116927,S2-180265553,S2-201976425
1878277      S1-189408085                                                NaN
993966       S1-797774718                          S2-151469064,S3-994706509
565428       S1-729164458             S2-619382517,S2-949886625,S3-532795614


In [7]:
gt["match_count"] = (
    gt["matched_entity_ids"]
    .fillna("")
    .str.split(",")
    .str.len()
)

display(
    gt[[
        "source1_entity_id",
        "matched_entity_ids",
        "match_count"
    ]].head(10)
)

,source1_entity_id,matched_entity_ids,match_count
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129...",5
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843...",4
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467",3
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215",3
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280...",6
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762...",4
6,S1-318373630,"S2-660036492,S3-804600254",2
7,S1-86989137,"S3-274817120,S3-312496301",2
8,S1-29845983,"S2-648035184,S3-588502663",2
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439",3


In [8]:
gt["match_count"] = (
    gt["matched_entity_ids"]
    .fillna("")
    .apply(lambda x: 0 if x == "" else len(x.split(",")))
)

display(
    gt[[
        "source1_entity_id",
        "matched_entity_ids",
        "match_count"
    ]].head(10)
)

,source1_entity_id,matched_entity_ids,match_count
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129...",5
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843...",4
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467",3
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215",3
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280...",6
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762...",4
6,S1-318373630,"S2-660036492,S3-804600254",2
7,S1-86989137,"S3-274817120,S3-312496301",2
8,S1-29845983,"S2-648035184,S3-588502663",2
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439",3


In [9]:
match_distribution = (
    gt["match_count"]
    .value_counts()
    .sort_index()
)

display(match_distribution)

match_count
0     123247
1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37
Name: count, dtype: int64

In [10]:
print("Total S1 entities:", len(gt))
print("Total with no matches:", (gt["match_count"] == 0).sum())
print("Total with at least one match:", (gt["match_count"] > 0).sum())
print("Maximum matches:", gt["match_count"].max())
print("Mean matches:", gt["match_count"].mean())

Total S1 entities: 2206821
Total with no matches: 123247
Total with at least one match: 2083574
Maximum matches: 11
Mean matches: 3.4612526344456573


In [11]:
from sklearn.model_selection import train_test_split

train_gt, valid_gt = train_test_split(
    gt,
    test_size=0.20,
    random_state=42,
    stratify=gt["match_count"]
)

print("Train:", train_gt.shape)
print("Validation:", valid_gt.shape)

Train: (1765456, 3)
Validation: (441365, 3)


In [12]:
print("Train match-count distribution:")
print(train_gt["match_count"].value_counts(normalize=True).sort_index())

print("\nValidation match-count distribution:")
print(valid_gt["match_count"].value_counts(normalize=True).sort_index())

Train match-count distribution:
match_count
0     0.055848
1     0.053995
2     0.170023
3     0.240546
4     0.219372
5     0.145891
6     0.074708
7     0.028986
8     0.008465
9     0.001905
10    0.000242
11    0.000017
Name: proportion, dtype: float64

Validation match-count distribution:
match_count
0     0.055847
1     0.053994
2     0.170025
3     0.240545
4     0.219372
5     0.145893
6     0.074709
7     0.028987
8     0.008465
9     0.001905
10    0.000242
11    0.000016
Name: proportion, dtype: float64


In [13]:
train_s1_ids = set(train_gt["source1_entity_id"])
valid_s1_ids = set(valid_gt["source1_entity_id"])

overlap = train_s1_ids & valid_s1_ids

print("Train S1 entities:", len(train_s1_ids))
print("Validation S1 entities:", len(valid_s1_ids))
print("Overlap:", len(overlap))

Train S1 entities: 1765456
Validation S1 entities: 441365
Overlap: 0


In [14]:
assert len(overlap) == 0
assert len(train_s1_ids) + len(valid_s1_ids) == len(gt)

print("✓ No S1 entity appears in both train and validation.")
print("✓ Every S1 entity belongs to exactly one split.")

✓ No S1 entity appears in both train and validation.
✓ Every S1 entity belongs to exactly one split.


In [15]:
split_dir = INTERIM_DATA_DIR / "splits"
split_dir.mkdir(parents=True, exist_ok=True)

train_s1 = train_gt[["source1_entity_id"]].copy()
valid_s1 = valid_gt[["source1_entity_id"]].copy()

train_s1.to_csv(
    split_dir / "train_s1_ids.tsv",
    sep="\t",
    index=False
)

valid_s1.to_csv(
    split_dir / "valid_s1_ids.tsv",
    sep="\t",
    index=False
)

print("Saved:")
print(split_dir / "train_s1_ids.tsv")
print(split_dir / "valid_s1_ids.tsv")

Saved:
d:\amzn_ml_challenge\data\interim\splits\train_s1_ids.tsv
d:\amzn_ml_challenge\data\interim\splits\valid_s1_ids.tsv
